# EX2D2 actuator — 2×2 cross-evaluation + temperature robustness (GPU)

Forward-solves **every** frozen actuator design under **all four** evaluation physics — the full
2×2 factorial of constitutive model × property temperature dependence — **and at every evaluation
temperature** `T_D ∈ {673, 873, 1073} K` (the design's own training `T_D` is always included):

| evaluation cell | constitutive | material properties |
|---|---|---|
| `linelas + const` | small strain | **fully constant**: room-temperature (293 K) base values |
| `linelas + tdep` | small strain | variable, `T`-dependent |
| `hencky + const` | finite-strain quadratic-Hencky | **fully constant**: room-temperature (293 K) base values |
| `hencky + tdep` **(reference)** | finite-strain quadratic-Hencky | variable, `T`-dependent |

The `const` cell uses the CFG base constants with **no temperature correction anywhere** —
`κ_i(293 K)`, `E_i(293 K)`, and the constant base CTE, `ε_th = α_i·(T − T_inf)` — i.e. the
`tdep_props_on = False` model of the training scripts and the *constant properties* cell of the
paper's 2×2 factorial. It does not depend on `T_D` at all (only the thermal BC does).

N designs × 3 evaluation `T_D` × 4 cells (per design and `T_D`: 2 thermal + 4 mechanical solves —
the thermal problem is one-way coupled and depends only on the property model, so each temperature
field is solved once and shared by both constitutive laws).

The folder name `Run_<seed>_<const>_<TD>` only records which physics and temperature the design
was **optimized** under (its *native* cell/`T_D`, together with `PROP_FOR_CONST`); every design is
solved in all cells at all evaluation temperatures regardless. All % effects in the tables are
normalized by the reference cell `u_ref = u(hencky, tdep)`, which is also the single *judge* in
the robustness tables (sections F–G) and figures.

The design is frozen at the **categorical material map** (`topo_material_id.csv`: 0 void, 1 Ti,
2 Cu, 3 Fe), i.e. the manufactured design, one-hot phase weights, SIMP at the final `p = pf = 3`.

**What is reproduced from the training script** (same mesh, same weak forms, same ports):

- unigrid `200 × 100` over the full `[0,500] × [0,250]` rectangle (no notch) →
  20 000 elements / 20 301 nodes / 40 602 dofs, full `2×2` Gauss;
- thermal: `0.5 ∫ κ̂|∇T|² t dV − ∫ ŝ T t dV`, Picard on `κ(T)` (single linear solve for `const`),
  Dirichlet `T = T_D` on the **left** edge only (confirmed: *"Left edge only"*);
- mechanical: `∫ ψ(H; C₁,C₂,ε_th) t dV + ½ K_out u_port²`, **no external work**. **Left + top
  edges clamped, bottom edge roller `v = 0`, `u` free** (confirmed: *"the left and top edges are
  clamped. the bottom is v=0, u = free"*), right edge free (paper Fig. 2(a));
- `u_out` = **horizontal** displacement of the bottom-right port node `(500, 0)`
  (`uout_comp = 0`); the port sits **on** the rollered bottom edge, so `v = 0` there by
  construction and the stroke is purely horizontal — exactly the trained objective.

**Solver — classic FEM, not energy minimization.** `linelas` is one linear solve (the potential
is quadratic, so it is exact). `hencky` is incremental Newton–Raphson on the residual
`R(u) = ∂Π/∂u = 0` with the consistent tangent (geometric term retained) and full Newton steps.
The thermal eigenstrain is applied in increments `ε_th → s·ε_th`, `s: 0 → 1`. That
incrementation is **not** cosmetic: at `u = 0` under the full thermal load the solid carries its
whole *compressive* thermal pre-stress, `K_T` is **indefinite**, and a Newton step taken from the
undeformed state diverges. (Same indefiniteness the training script's direct adjoint handles with
an indefinite-safe `LDL^T`.) As `s → 0` the tangent is the positive-definite material tangent, so
the path starts well-posed.

**Assumptions**: single-node output spring at `(500, 0)` (zero snap offset, `K_out = 2e-3`), no
mechanical input load, `hs`/`hv` unused (pure Dirichlet + adiabatic + volumetric source, as in
`calculate_TO_loss`). Fix in `Mesh.__init__` / `CFG` if wrong — one place each.

**Expected data layout**: a zip that extracts to a folder of `Run_<seed>_<const>_<TD>` design
folders, each containing `topo_material_id.csv` in the COMSOL-spreadsheet format written by
`viz_unigrid_TiCuFe_tdep.py` (2.5 µm cell centres, 20 000 rows). The next cell auto-detects the
folder, so the zip's own name does not matter.

## 1 · GPU + sparse direct backend

Runtime → Change runtime type → **T4 GPU** (any GPU is plenty; this is a 37 k-dof problem).

The solver tries `torch_sla` + **cuDSS** first (`LDL^T`, `matrix_type='symmetric'` — the safe factorization for the indefinite thermally pre-stressed tangent), exactly as the training script's `_sparse_direct_solve` does; then CuPy; then SciPy/SuperLU on the host.

**Every backend must pass a numerical probe** — a small symmetric *indefinite* system with a known solution — before it is selected, and any runtime failure permanently demotes it to the next one. So a broken cuDSS is never fatal and can never silently return a wrong answer; you just fall back and lose some speed. All three backends give the same numbers.

**The cuDSS breakage, precisely.** `torch-sla[cudss]` depends on `nvmath-python[cu12]` *unpinned*, which now resolves to **nvmath-python 1.0.0**. Between 0.9.0 and 1.0.0 NVIDIA inserted a new `offset_type` parameter at position 7 of `cudss.matrix_create_csr`:

```
<= 0.9.0 :  (nrows, ncols, nnz, row_start, row_end, col_indices, values,
             index_type, value_type, mtype, mview, index_base)          # 12
>= 1.0.0 :  (nrows, ncols, nnz, row_start, row_end, col_indices, values,
             offset_type, index_type, value_type, mtype, mview, index_base)  # 13
```

`torch-sla` 0.3.x still calls the 12-argument form — hence `TypeError: matrix_create_csr() takes exactly 13 positional arguments (12 given)`. Pinning `nvmath-python==0.9.0` fixes it at the source (that extra also pulls `nvidia-cudss-cu12==0.7.*` and the CUDA runtime libs). If some environment forces 1.0 anyway, the module also bridges the call in-process by duplicating `index_type` into the new `offset_type` slot — exact rather than guessed, because `torch-sla` builds the row offsets and column indices from the same `.int()` cast.

In [ ]:
# CLEAN cuDSS INSTALL.  Run this FIRST, before anything imports nvmath/torch_sla.
#
# Why the pin: torch-sla[cudss] requires `nvmath-python[cu12]` UNPINNED, which
# today resolves to nvmath-python 1.0.0.  nvmath 1.0.0 inserted a new
# `offset_type` argument at position 7 of cudss.matrix_create_csr (12 args ->
# 13); torch-sla 0.3.x still calls the 12-argument form, hence
#     TypeError: matrix_create_csr() takes exactly 13 positional arguments
# 0.9.0 is the newest nvmath-python whose signature torch-sla matches, and its
# [cuXX] extra pulls nvidia-cudss-cuXX==0.7.* plus every CUDA runtime library.
import sys, subprocess
!nvidia-smi -L || echo "NO GPU -- Runtime > Change runtime type > T4 GPU"

import torch
CU = (torch.version.cuda or "12").split(".")[0]          # "12" or "13"
print(f"torch {torch.__version__}  cuda {torch.version.cuda}  -> nvmath extra cu{CU}")

def pip(*a):
    print("$ pip install", " ".join(a))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a])

pip("--upgrade", f"nvmath-python[cu{CU}]==0.9.0")   # bindings + cuDSS + CUDA libs
pip("torch-sla")                                    # solver dispatch layer

import importlib.metadata as md
nv, ts = md.version("nvmath-python"), md.version("torch-sla")
print(f"\nnvmath-python {nv} | torch-sla {ts}")
if int(nv.split('.')[0]) >= 1:
    print("!! nvmath-python >= 1.0 is still what pip resolved.\n"
          "   The in-process signature bridge will handle it, but the clean\n"
          "   route is: Runtime > Restart session, then re-run this cell.")
if "nvmath" in sys.modules or "torch_sla" in sys.modules:
    print("!! nvmath/torch_sla were already imported in this session.\n"
          "   Runtime > Restart session before continuing, or the OLD version\n"
          "   stays loaded.")

## 2 · Upload the data

Upload the **processed designs zip** (any name — the next cell auto-detects the folder containing
the `Run_*` design directories). Optionally also upload the **training script** (`ex1_jul27.py`,
`EX2D1_*.py` or `EX2D2_*.py`) — if present, the check cell asserts the constitutive block embedded
here is *code-identical* to the training script's (docstrings excluded), so the port cannot
silently drift.

In [ ]:
import os, glob, zipfile
from google.colab import files

def find_root():
    hits = {os.path.dirname(p)
            for p in glob.glob('**/Run_*/topo_material_id.csv', recursive=True)}
    roots = sorted({os.path.dirname(h) for h in hits})
    return roots[0] if roots else None

ROOT = find_root()
if ROOT is None:
    up = files.upload()                  # pick the processed zip (+ training .py, optional)
    for name in up:
        if name.endswith('.zip'):
            zipfile.ZipFile(name).extractall('.')
    ROOT = find_root()
assert ROOT, 'no Run_*/topo_material_id.csv found after extraction'
runs = sorted(glob.glob(os.path.join(ROOT, 'Run_*')))
print('ROOT =', ROOT, '|', len(runs), 'design folders')
print(*[os.path.basename(r) for r in runs], sep='\n')

## 3 · The solver

Self-contained. The constitutive block (quadratic-Hencky log-strain law, polynomial property helpers, `_elem_energy_dofs`) is copied **verbatim** from the training script.

In [ ]:
%%writefile ex2d2_forward.py
#!/usr/bin/env python3
# =====================================================================
# EX2D1 GRIPPER / EX2D2 ACTUATOR -- 2x2 CROSS-EVALUATION +
# TEMPERATURE ROBUSTNESS (GPU).  Select the example with EXAMPLE below.
# ---------------------------------------------------------------------
# Takes each converged multi-material design as its CATEGORICAL material
# map (topo_material_id.csv: 0 void, 1 Ti, 2 Cu, 3 Fe) and forward-solves
# the SAME frozen design under ALL FOUR evaluation physics
#
#         {linelas, hencky}  x  {const, tdep}
#
# i.e. small-strain vs finite-strain quadratic-Hencky, crossed with
# FULLY CONSTANT properties vs fully T-dependent properties, at the
# design's own TD (read from the folder name Run_<seed>_<const>_<TD>).
# 'const' = the room-temperature (T_ref = 293 K) base values with NO
# temperature correction anywhere and eps_th = alpha_base*(T - T_inf) --
# the tdep_props_on = False model of the training scripts, i.e. the
# 'constant properties' cell of the paper's 2x2 factorial.  It does not
# depend on TD at all (only the thermal BC does).
# No optimization, no GP/NN -- the pure FE problem the DEM was
# approximating, on the SAME unigrid mesh, weak forms, BCs and ports.
#
# TEMPERATURE ROBUSTNESS: on top of the physics cross, every design is
# evaluated at EVERY TD in EVAL_TDS = {673, 873, 1073} K (its own TD is
# always included), i.e. design x eval-TD x cell.  The folder name's
# <const>/<TD> only record which physics and temperature the design was
# OPTIMIZED under (its 'native' cell/TD, together with PROP_FOR_CONST);
# every design is solved in all cells at all eval TDs regardless.  The
# thermal problem is one-way coupled and depends only on the property
# model, so per design and eval TD the two temperature fields are solved
# once each and shared by both constitutive laws: 2 thermal + 4
# mechanical solves per (design, TD).
#
# EXAMPLE GEOMETRY / BCs (both confirmed against the training scripts
# and past sessions; T = TD Dirichlet on the LEFT edge only for both):
#   EX2D1 gripper : [0,500]x[0,250] minus the top-right 100x100 notch;
#                   LEFT + BOTTOM clamped, TOP roller v=0 up to
#                   x = GRIP_FREE_X0 = 400; port = jaw tip (500, 150),
#                   u_out = VERTICAL stroke (uout_comp = 1).
#   EX2D2 actuator: full [0,500]x[0,250] rectangle; LEFT + TOP clamped,
#                   BOTTOM roller v = 0 (u free); port = bottom-right
#                   corner (500, 0), u_out = HORIZONTAL stroke
#                   (uout_comp = 0).
#
# PHYSICS -- transcribed from calculate_TO_loss in the training script:
#
#   THERMAL    Pi_T = 0.5 int kappa_hat |grad T|^2 t dV - int s_hat T t dV
#              kappa_hat = sum_i kappa_i f_k,i(dT) w_i^p     (SIMP, p = pf = 3)
#              s_hat     = sum_i s_i w_i^p
#              Properties are evaluated at the DETACHED current T -- the
#              loop's Picard treatment -- so the converged fixed point
#              solves the nonlinear weak form
#                  int kappa(T) grad T . grad w = int s w .
#              Dirichlet  T = TD  on the LEFT edge (x = xmin).
#
#   MECHANICAL Pi_u = int psi(H; C1, C2, eps_th) t dV + 0.5 K_out v_port^2
#              external_work == 0: the device is purely thermally actuated.
#              psi:  'linelas' -> W_density_linear_thermal
#                    'hencky'  -> three-term Wang interpolation, with
#                                 gamma = H_beta(rho_solid) (exactly 0 or 1
#                                 on a binary material map)
#              E_hat  = sum_i E_i f_E,i(dT) w_i^p
#              C1 = E_hat/(1-nu^2),  C2 = E_hat/(2(1+nu))
#              eps_th = sum_i w_i alpha_i [F_a,i(dT) - F_a,i(dT0)]  (exact
#                       integral of the instantaneous CTE)
#              Dirichlet (paper Fig. 2(a) / _ex2d2_adjoint_fixed_dofs):
#                  LEFT (x = xmin) and TOP (y = ymax) edges CLAMPED,
#                  BOTTOM edge (y = ymin) ROLLER v = 0 (u free),
#                  right edge FREE.
#
#   u_out = u at the output-port node = the bottom-right corner
#           (500, 0); MP['uout_comp'] = 0 -> HORIZONTAL stroke.
#
# SOLVER -- classic FEM, not energy minimization:
#   * linelas: ONE linear solve (the potential is quadratic; exact).
#   * hencky : incremental Newton-Raphson on the residual R(u) = dPi/du = 0,
#              consistent tangent with the geometric term retained, full
#              Newton steps, thermal eigenstrain applied in increments
#              eps_th -> s*eps_th, s: 0 -> 1, increment halved on failure.
#              The incrementation is NOT cosmetic: at u = 0 under the FULL
#              thermal load the solid carries its whole COMPRESSIVE thermal
#              pre-stress, K_T is INDEFINITE, and a Newton step taken from
#              the undeformed state diverges.  As s -> 0 the tangent is the
#              positive-definite material tangent, so the path starts
#              well-posed and NR tracks the equilibrium branch.  (The
#              training script hits the same indefiniteness -- it is why its
#              direct adjoint uses an indefinite-safe LDL^T.)
#
# LINEAR ALGEBRA: sparse direct, float64, backend auto-selected --
#   torch_sla + cuDSS (LDL^T, symmetric-indefinite safe)  ->  cupy  ->  scipy.
#
# The constitutive kernels below are copied VERBATIM from the training
# script (quadratic-Hencky log-strain law + the polynomial helpers +
# _elem_energy_dofs).  check_verbatim() re-extracts them from the training
# file, if it is present, and asserts textual identity so the port cannot
# silently drift.
# =====================================================================
from __future__ import annotations

import argparse
import csv
import json
import math
import os
import re
import time
from pathlib import Path

import numpy as np
import torch

torch.backends.cuda.matmul.allow_tf32 = False   # fp64 assembly; keep exact
DT = torch.float64


# =====================================================================
# ---------------- VERBATIM from the training script ------------------
# (quadratic-Hencky constitutive law, polynomial property helpers, and
#  the element energy used for the consistent tangent)
# =====================================================================
def _f_atanh(q):
    """f(q) = atanh(sqrt(q))/sqrt(q), smooth on [0,1); double-where keeps the
    unused branch NaN-free in the backward pass."""
    small = q < 1e-8
    q_safe = torch.where(small, torch.full_like(q, 0.5), q)
    sq = torch.sqrt(q_safe)
    exact = torch.atanh(sq) / sq
    series = 1.0 + q / 3.0 + q * q / 5.0 + q**3 / 7.0 + q**4 / 9.0
    return torch.where(small, series, exact)


Q_MAX = 1.0 - 1e-3


def _hencky_eps_from_C(C11, C22, C12):
    """In-plane material Hencky strain eps_H = 1/2 log(C) from the right
    Cauchy-Green components (C11, C22, C12)."""
    I1 = C11 + C22
    detC = (C11 * C22 - C12 * C12).clamp_min(1e-12)
    disc = (C11 - C22) ** 2 + 4.0 * C12 * C12
    q = (disc / (I1 * I1)).clamp(0.0, Q_MAX)
    beta = _f_atanh(q) / I1
    tr_eps = 0.5 * torch.log(detC)
    alpha = 0.5 * tr_eps - 0.5 * beta * I1
    eps11 = alpha + beta * C11
    eps22 = alpha + beta * C22
    gam12 = 2.0 * beta * C12
    return eps11, eps22, gam12


def hencky_strain_2d(H):
    """In-plane material Hencky strain eps_H = 1/2 log(F^T F).
    H : [...,2,2] displacement gradient dU_i/dX_j (total Lagrangian)."""
    F11 = 1.0 + H[..., 0, 0]; F12 = H[..., 0, 1]
    F21 = H[..., 1, 0];       F22 = 1.0 + H[..., 1, 1]
    C11 = F11 * F11 + F21 * F21
    C22 = F12 * F12 + F22 * F22
    C12 = F11 * F12 + F21 * F22
    return _hencky_eps_from_C(C11, C22, C12)


def W_density_hencky_thermal(H, C1, C2, nu, eps_th):
    """Plane-stress quadratic-Hencky strain-energy density with an isotropic
    log-thermal eigenstrain eps_th subtracted from the normal log-strains."""
    e11, e22, g12 = hencky_strain_2d(H)
    e11 = e11 - eps_th
    e22 = e22 - eps_th
    t11 = C1 * (e11 + nu * e22)
    t22 = C1 * (nu * e11 + e22)
    t12 = C2 * g12
    return 0.5 * (t11 * e11 + t22 * e22 + t12 * g12)


def W_density_linear_thermal(H, C1, C2, nu, eps_th):
    """Small-strain plane-stress energy (void branch of the Wang
    interpolation) with the same isotropic thermal eigenstrain subtracted."""
    e11 = H[..., 0, 0] - eps_th
    e22 = H[..., 1, 1] - eps_th
    g12 = H[..., 0, 1] + H[..., 1, 0]
    t11 = C1 * (e11 + nu * e22)
    t22 = C1 * (nu * e11 + e22)
    t12 = C2 * g12
    return 0.5 * (t11 * e11 + t22 * e22 + t12 * g12)


def _poly_eval(coef, x):
    """sum_k coef[k] * x**k  (Horner; x tensor or float)."""
    y = 0.0
    for c in reversed(coef):
        y = y * x + c
    return y


def _poly_antideriv(coef, x):
    """Antiderivative with zero constant:  sum_k coef[k]/(k+1) * x**(k+1)."""
    y = 0.0
    for k in range(len(coef) - 1, -1, -1):
        y = y * x + coef[k] / (k + 1)
    return y * x


def _elem_energy_dofs(d, BT_e, g_e, c1_e, c2_e, eth_e, wdetJ_e, nu, thickness,
                      linelas):
    """Interpolated three-term Wang energy of ONE element as a function of its
    8 nodal DOFs d = [u0,v0,u1,v1,u2,v2,u3,v3] (vmapped over elements; the
    per-element Hessian of this scalar is the exact consistent tangent K_e)."""
    un = d.view(4, 2)
    H = torch.matmul(BT_e, un).transpose(-1, -2)              # [N_int, 2, 2]
    if linelas:
        W = W_density_linear_thermal(H, c1_e, c2_e, nu, eth_e)
    else:
        W = (W_density_hencky_thermal(g_e * H, c1_e, c2_e, nu, eth_e)
             + W_density_linear_thermal(H, c1_e, c2_e, nu, eth_e)
             - W_density_linear_thermal(g_e * H, c1_e, c2_e, nu, eth_e))
    return (W * wdetJ_e).sum() * thickness
# ------------------- end verbatim block ------------------------------


# =====================================================================
# Config -- copied from the ex1_jul27.py launch block
# =====================================================================
CFG = dict(
    T_inf=293.0,
    s=[0.0, -4.5e-8, -4.5e-8, -4.5e-8],
    kappa=[1.0e-8, 2.19e-5, 40e-5, 6e-5],       # W/(K um)
    alpha=[1.2e-5, 0.86e-5, 1.7e-5, 1.2e-5],    # 1/K
    E=[1e-5, 0.115, 0.128, 0.2],                # GPa/1000
    D=[0.0, 4.506, 8.96, 7.8],
    P=[0.0, 3.0, 2.0, 1.0],
    K_out=2e-3, f_out=0.1,
    nu=0.31, pf=3.0, thickness=15.0,
    xmin=0.0, xmax=500.0, ymin=0.0, ymax=250.0,
    unigrid=(200, 100),
    gamma_beta=100.0, gamma_eta=0.10,
    prop_T_ref=293.0, prop_T_valid=(293.0, 1100.0),
    kappa_T_coef=[[1.0],
                  [1.0, -8.32894e-4, 2.02144e-6, -1.87281e-9, 7.42533e-13],
                  [1.0, -1.72205e-4],
                  [1.0, -5.38540e-4, -5.82423e-7, 5.41279e-10]],
    alpha_T_coef=[[1.0],
                  [1.0, 4.50136e-4, -1.09250e-7],
                  [1.0, 5.18040e-4, -3.84006e-7, 4.00361e-10],
                  [1.0, 9.45641e-4, -2.26320e-6, 4.44551e-9, -3.06905e-12]],
    E_T_coef=[[1.0],
              [1.0, -5.60051e-4, 1.04476e-7, -2.11509e-10],
              [1.0, -2.46040e-4, -4.35741e-7, 2.03154e-10],
              [1.0, -3.20222e-4, -1.71611e-7, -1.68928e-10]],
)
PHASE_NAMES = ['void', 'Ti', 'Cu', 'Fe']

# ---------------------------------------------------------------------
# EXAMPLE SWITCH -- the one line that differs between the two notebooks
# ---------------------------------------------------------------------
EXAMPLE = 'EX2D2'

CFG['example'] = EXAMPLE
if EXAMPLE == 'EX2D1':
    CFG.update(uout_comp=1, port_xy=(500.0, 150.0),
               GRIP_FREE_X0=400.0, NOTCH_DEPTH=100.0)
else:                                   # EX2D2 actuator
    CFG.update(uout_comp=0, port_xy=(500.0, 0.0))
EX_TITLE = {'EX2D1': 'EX2D1 GRIPPER', 'EX2D2': 'EX2D2 ACTUATOR'}

# the factorial pairing each design family represents (marks the 'native'
# cell in tables/figures only; the sweep always evaluates all four cells)
PROP_FOR_CONST = {'linelas': 'const', 'hencky': 'tdep'}

# the 2x2 evaluation grid, in fixed report/plot order
EVAL_CELLS = [('linelas', 'const'), ('linelas', 'tdep'),
              ('hencky', 'const'), ('hencky', 'tdep')]


def cell_tag(const, pe):
    return f"{const}_{pe}"


def cell_label(const, pe):
    return f"{const} + {'const props' if pe == 'const' else '$T$-dep props'}"


# evaluation temperatures for the robustness sweep (a design's own TD is
# always solved even if it is missing from this list)
EVAL_TDS = [673.0, 873.0, 1073.0]

# the cell used as the single judge in the robustness tables/figures
REF_CELL = ('hencky', 'tdep')


# =====================================================================
# Sparse direct solver -- cuDSS / cupy / scipy, float64
# =====================================================================
def _try_patch_cudss_binding(verbose=True):
    """Bridge the torch_sla <-> nvmath-python signature break.

    nvmath-python <= 0.9.0:
        matrix_create_csr(nrows, ncols, nnz, row_start, row_end, col_indices,
                          values, index_type, value_type, mtype, mview,
                          index_base)                                 # 12 args
    nvmath-python >= 1.0.0 inserted a NEW `offset_type` at position 7:
        matrix_create_csr(..., values, offset_type, index_type, value_type,
                          mtype, mview, index_base)                   # 13 args
    torch_sla 0.3.x still calls the 12-argument form, so on nvmath >= 1.0 it
    dies with
        TypeError: matrix_create_csr() takes exactly 13 positional arguments
                   (12 given)

    THE PREFERRED FIX IS THE PIN  `nvmath-python[cuXX]==0.9.0`  (see the
    install cell).  This wrapper is the in-process fallback: on that specific
    TypeError it re-issues the call with arg[7] DUPLICATED into the new
    `offset_type` slot.  That is exact, not a guess -- torch_sla builds the
    row offsets and the column indices from the same `.int()` cast
        crow = A_csr.crow_indices().int();  ccol = A_csr.col_indices().int()
    and passes CUDA_R_32I as `index_type`, so `offset_type == index_type`.

    SparseDirect._probe still verifies the result numerically against a known
    solution of a symmetric INDEFINITE system before the backend is used, so
    even a wrong bridge can only disable cuDSS, never corrupt a solve.
    """
    try:
        from nvmath.bindings import cudss
    except Exception as e:
        if verbose:
            print(f"[solver] nvmath cudss bindings not importable ({type(e).__name__})")
        return False
    if getattr(cudss.matrix_create_csr, '_argpatched', False):
        return True
    orig = cudss.matrix_create_csr

    def patched(*a, **kw):
        try:
            return orig(*a, **kw)
        except TypeError as e:
            if 'positional argument' not in str(e) or len(a) < 8:
                raise
            # insert offset_type := index_type (a[7]) at position 7
            return orig(*a[:7], a[7], *a[7:], **kw)
    patched._argpatched = True
    ok = False
    try:
        cudss.matrix_create_csr = patched
        ok = True
    except Exception:
        pass
    try:    # in case torch_sla imported the symbol by value
        import torch_sla.backends.nvmath_backend as nb
        if hasattr(nb, 'matrix_create_csr'):
            nb.matrix_create_csr = patched
        if hasattr(nb, 'cudss'):
            nb.cudss.matrix_create_csr = patched
        ok = True
    except Exception:
        pass
    if verbose and ok:
        print("[solver] patched nvmath cudss.matrix_create_csr arity "
              "(will be numerically probed before use)")
    return ok


class SparseDirect:
    """Symmetric, possibly INDEFINITE, sparse direct solve in float64.

    The tangent here IS indefinite (compressive thermal pre-stress), so the
    preferred backend is torch_sla + cuDSS with ``matrix_type='symmetric'``
    -> LDL^T, exactly as the training script's _sparse_direct_solve does.
    CuPy and SciPy/SuperLU are fallbacks; all three give the same answer.

    Every backend must pass a numerical PROBE (a small symmetric indefinite
    system with a known solution) before it is selected, and any runtime
    failure permanently demotes it to the next one.  Nothing is used on the
    strength of "it imported".
    """
    ORDER = ['torch_sla', 'cupy', 'scipy']

    def __init__(self, backend='auto', method='ldlt', verbose=True):
        self.method = method
        self.verbose = verbose
        self._scipy_pat = {}
        cands = list(self.ORDER) if backend == 'auto' else [backend]
        self.backend = None
        for b in cands:
            if b == 'torch_sla' and torch.cuda.is_available():
                _try_patch_cudss_binding(verbose)
            ok, why = self._probe(b)
            if ok:
                self.backend = b
                break
            if verbose:
                print(f"[solver] backend '{b}' rejected: {why}")
        if self.backend is None:
            raise RuntimeError("no working sparse direct backend")
        if verbose:
            print(f"[solver] sparse direct backend = {self.backend} "
                  f"(method={self.method}, float64)")

    # -- probe ---------------------------------------------------------
    def _probe(self, backend):
        if backend in ('torch_sla', 'cupy'):
            if not torch.cuda.is_available():
                return False, 'no CUDA device'
            try:
                __import__('torch_sla' if backend == 'torch_sla' else 'cupy')
            except Exception as e:
                return False, f'import failed ({type(e).__name__})'
        dev = torch.device('cuda' if backend in ('torch_sla', 'cupy') else 'cpu')
        A = torch.tensor([[4., -1., 0., 0., 0., 0.],
                          [-1., 3., -1., 0., 0., 0.],
                          [0., -1., -2., -1., 0., 0.],   # negative pivot:
                          [0., 0., -1., 5., -1., 0.],    # genuinely indefinite
                          [0., 0., 0., -1., 4., -1.],
                          [0., 0., 0., 0., -1., 3.]], dtype=DT, device=dev)
        n = A.shape[0]
        x_ex = torch.arange(1., n + 1., dtype=DT, device=dev)
        b = A @ x_ex
        idx = A.nonzero()
        rows, cols = idx[:, 0].contiguous(), idx[:, 1].contiguous()
        vals = A[rows, cols].contiguous()
        try:
            x = self._raw(backend, rows, cols, vals, b, n, key=None)
        except Exception as e:
            return False, f'{type(e).__name__}: {str(e).splitlines()[0][:110]}'
        try:
            err = float((x.to(dev).reshape(-1) - x_ex).abs().max()
                        / x_ex.abs().max())
        except Exception as e:
            return False, f'bad return ({type(e).__name__})'
        if not np.isfinite(err) or err > 1e-8:
            return False, f'probe error {err:.2e} (backend answered, wrongly)'
        return True, ''

    # -- backends ------------------------------------------------------
    def _raw(self, backend, rows, cols, vals, b, n, key=None):
        if backend == 'torch_sla':
            import torch_sla
            try:
                return torch_sla.spsolve(vals.double(), rows, cols, (n, n),
                                         b.double(), backend='cudss',
                                         method=self.method,
                                         matrix_type='symmetric',
                                         is_symmetric=True)
            except TypeError as e:
                # only an OLD torch_sla SIGNATURE justifies the short retry;
                # a TypeError from inside the cuDSS bindings must propagate
                if 'unexpected keyword' not in str(e):
                    raise
                return torch_sla.spsolve(vals.double(), rows, cols, (n, n),
                                         b.double(), backend='cudss',
                                         method=self.method)
        if backend == 'cupy':
            import cupy as cp
            import cupyx.scipy.sparse as csp
            import cupyx.scipy.sparse.linalg as csla
            r = cp.from_dlpack(rows.to(torch.int32).contiguous().detach())
            c = cp.from_dlpack(cols.to(torch.int32).contiguous().detach())
            v = cp.from_dlpack(vals.double().contiguous().detach())
            bb = cp.from_dlpack(b.double().contiguous().detach())
            A = csp.coo_matrix((v, (r, c)), shape=(n, n)).tocsr()
            return torch.from_dlpack(csla.spsolve(A, bb))
        # ---- scipy / SuperLU, with a CACHED CSC pattern -------------------
        from scipy.sparse.linalg import splu
        pat = self._scipy_pat.get(key) if key else None
        if pat is None or pat['nnz_raw'] != int(rows.numel()):
            r = rows.cpu().numpy().astype(np.int64)
            c = cols.cpu().numpy().astype(np.int64)
            lin = c * n + r                       # column-major -> CSC order
            order = np.argsort(lin, kind='stable')
            uniq, first = np.unique(lin[order], return_index=True)
            counts = np.zeros(n + 1, dtype=np.int64)
            np.add.at(counts, (uniq // n) + 1, 1)
            pat = dict(order=order, seg=first,
                       indices=(uniq % n).astype(np.int32),
                       indptr=np.cumsum(counts).astype(np.int32),
                       nnz_raw=int(rows.numel()))
            if key:
                self._scipy_pat[key] = pat
        from scipy.sparse import csc_matrix
        data = np.add.reduceat(vals.double().cpu().numpy()[pat['order']],
                               pat['seg'])
        A = csc_matrix((data, pat['indices'], pat['indptr']), shape=(n, n))
        x = splu(A).solve(b.double().cpu().numpy())
        return torch.as_tensor(x, dtype=DT, device=b.device)

    # -- public --------------------------------------------------------
    def solve(self, rows, cols, vals, b, n, key=None):
        """A x = b for COO (rows, cols, vals) of shape (n, n).  Torch in/out.
        `key` names a REUSED sparsity pattern (the SciPy path caches the CSC
        structure under it and only re-reduces the values)."""
        while True:
            try:
                x = self._raw(self.backend, rows, cols, vals, b, n, key=key)
                if bool(torch.isfinite(x).all()):
                    return x.to(b.device).reshape(-1)
                raise RuntimeError('non-finite solution')
            except Exception as e:
                nxt = self.ORDER[self.ORDER.index(self.backend) + 1:]
                if not nxt:
                    raise
                if self.verbose:
                    print(f"[solver] '{self.backend}' failed at runtime "
                          f"({type(e).__name__}: {str(e).splitlines()[0][:90]});"
                          f" demoting to '{nxt[0]}'")
                self.backend = nxt[0]


def _apply_dirichlet(rows, cols, vals, n, fixed_mask, diag_scale):
    """Row/column elimination with a unit (scaled) diagonal on the fixed dofs.
    The RHS must already be zero there, so the solution is exactly 0 on them --
    correct for a homogeneous-increment Newton step and for the thermal
    correction field."""
    keep = (~fixed_mask[rows]) & (~fixed_mask[cols])
    fidx = torch.nonzero(fixed_mask, as_tuple=False).reshape(-1)
    r = torch.cat([rows[keep], fidx])
    c = torch.cat([cols[keep], fidx])
    v = torch.cat([vals[keep],
                   torch.full((fidx.numel(),), diag_scale,
                              device=vals.device, dtype=vals.dtype)])
    return r, c, v


# =====================================================================
# Mesh -- reproduces _build_unigrid_mesh_data for the gripper geometry
# =====================================================================
class Mesh:
    def __init__(self, keep_ji, dev, cfg=CFG):
        """keep_ji : bool [Ngy, Ngx] cell-keep mask (holes / notch -> False).
        Cells with no material-map entry are DELETED and unreferenced nodes
        dropped + reindexed, exactly as the unigrid build does."""
        Ngx, Ngy = cfg['unigrid']
        xmin, xmax = cfg['xmin'], cfg['xmax']
        ymin, ymax = cfg['ymin'], cfg['ymax']
        hx, hy = (xmax - xmin) / Ngx, (ymax - ymin) / Ngy
        self.dev, self.hx, self.hy = dev, hx, hy

        xs = np.linspace(xmin, xmax, Ngx + 1)
        ys = np.linspace(ymin, ymax, Ngy + 1)
        Xg, Yg = np.meshgrid(xs, ys, indexing='xy')     # node id = j*(Ngx+1)+i
        nodes = np.stack([Xg.reshape(-1), Yg.reshape(-1)], axis=1)
        Jg, Ig = np.meshgrid(np.arange(Ngy), np.arange(Ngx), indexing='ij')
        n0 = (Jg * (Ngx + 1) + Ig).reshape(-1)
        conn = np.stack([n0, n0 + 1, n0 + Ngx + 2, n0 + Ngx + 1], axis=1)

        conn = conn[keep_ji.reshape(-1)]                # same j-major order
        used = np.zeros(nodes.shape[0], dtype=bool)
        used[conn.reshape(-1)] = True
        new_id = np.full(nodes.shape[0], -1, dtype=np.int64)
        new_id[used] = np.arange(used.sum())
        nodes = nodes[used]
        conn = new_id[conn]

        # --- FULL 2x2 Gauss operators (identical to the unigrid build) ------
        gp = 1.0 / math.sqrt(3.0)
        pts = [(-gp, -gp), (gp, -gp), (gp, gp), (-gp, gp)]
        BT = np.zeros((4, 2, 4)); Nsh = np.zeros((4, 4))
        for k, (xi, eta) in enumerate(pts):
            Nsh[k] = np.array([(1 - xi) * (1 - eta), (1 + xi) * (1 - eta),
                               (1 + xi) * (1 + eta), (1 - xi) * (1 + eta)]) / 4
            dN_dxi = np.array([-(1 - eta), (1 - eta), (1 + eta), -(1 + eta)]) / 4
            dN_deta = np.array([-(1 - xi), -(1 + xi), (1 + xi), (1 - xi)]) / 4
            BT[k, 0, :] = dN_dxi * (2.0 / hx)
            BT[k, 1, :] = dN_deta * (2.0 / hy)

        t = lambda a, d=DT: torch.as_tensor(a, dtype=d, device=dev)
        self.nodes = t(nodes)
        self.conn = t(conn, torch.long)
        self.Ne = int(conn.shape[0]); self.Nn = int(nodes.shape[0])
        self.BT = t(BT)                                 # [4int, 2, 4]
        self.N = t(Nsh)                                 # [4int, 4]
        self.wdetJ = t(np.full(4, hx * hy / 4.0))       # weights = ones(4)
        self.elem_x = self.nodes[self.conn].mean(dim=1)
        self.elem_vol = t(np.full(self.Ne, hx * hy))

        # --- boundary sets ---------------------------------------------------
        # EX2D1 gripper (paper Fig. 2(b)): LEFT + BOTTOM clamped, TOP roller
        #   v = 0 up to x = GRIP_FREE_X0; notch faces / jaw ledge / right free.
        # EX2D2 actuator (paper Fig. 2(a)): LEFT + TOP clamped, BOTTOM roller
        #   v = 0 (u free) along the whole edge; right edge free.
        # Both: T = TD Dirichlet on the LEFT edge only.
        ex = cfg.get('example', 'EX2D1')
        x, y = nodes[:, 0], nodes[:, 1]
        tolx = 1e-4 * (x.max() - x.min()); toly = 1e-4 * (y.max() - y.min())
        left = x <= x.min() + tolx
        bottom = y <= y.min() + toly
        top = y >= y.max() - toly
        if ex == 'EX2D1':
            clamped = left | bottom
            roller = top & (x <= cfg['GRIP_FREE_X0'])
        else:                                    # EX2D2
            clamped = left | top
            roller = bottom
        fu = np.zeros(2 * self.Nn, dtype=bool)
        fu[2 * np.where(clamped)[0]] = True
        fu[2 * np.where(clamped)[0] + 1] = True
        fu[2 * np.where(roller)[0] + 1] = True
        self.fixed_u = t(fu, torch.bool)
        self.fixed_T = t(left, torch.bool)
        self.n_clamped = int(clamped.sum()); self.n_roller = int(roller.sum())

        # --- output port -----------------------------------------------------
        px, py = cfg['port_xy']
        d2 = (x - px) ** 2 + (y - py) ** 2
        self.port = int(np.argmin(d2))
        self.port_off = float(np.sqrt(d2[self.port]))

        # --- assembly index patterns (constant per mesh) --------------------
        c = self.conn
        self.rowsT = c.repeat_interleave(4, dim=1).reshape(-1)
        self.colsT = c.repeat(1, 4).reshape(-1)
        edof = torch.stack([2 * c[:, 0], 2 * c[:, 0] + 1,
                            2 * c[:, 1], 2 * c[:, 1] + 1,
                            2 * c[:, 2], 2 * c[:, 2] + 1,
                            2 * c[:, 3], 2 * c[:, 3] + 1], dim=1)
        self.edof = edof
        self.rowsU = edof.repeat_interleave(8, dim=1).reshape(-1)
        self.colsU = edof.repeat(1, 8).reshape(-1)


# =====================================================================
# Material interpolation (mirrors the tdep block of calculate_TO_loss)
# =====================================================================
def props(w, T_ip, cfg, prop_eval, TD):
    """w : [Ne, 4] phase weights (one-hot for a material map).
    T_ip : [Ne, 4int] temperature at the integration points.
    Returns kappa_hat, E_hat, eps_th, s_hat, clamp_frac."""
    p = cfg['pf']
    Tref = cfg['prop_T_ref']
    Tlo, Thi = cfg['prop_T_valid']
    T_inf = cfg['T_inf']
    dev = w.device
    kap_b = torch.as_tensor(cfg['kappa'], dtype=DT, device=dev)
    alp_b = torch.as_tensor(cfg['alpha'], dtype=DT, device=dev)
    E_b = torch.as_tensor(cfg['E'], dtype=DT, device=dev)
    s_b = torch.as_tensor(cfg['s'], dtype=DT, device=dev)
    n_ph = w.shape[1]
    w_p = w ** p

    dTp = (T_ip - Tref).clamp(Tlo - Tref, Thi - Tref)
    inw = (T_ip > Tlo) & (T_ip < Thi)
    dTp0 = float(min(max(T_inf, Tlo), Thi) - Tref)

    kap = torch.zeros_like(dTp); Eh = torch.zeros_like(dTp)
    eps = torch.zeros_like(dTp)
    if prop_eval == 'const':
        # FULLY CONSTANT properties: the room-temperature (T_ref = 293 K)
        # base values with NO temperature correction anywhere.  kappa_i and
        # E_i are the CFG base constants (the f-polynomials are normalised
        # to 1 at T_ref) and the eigenstrain uses the constant base CTE:
        #     eps_th(x) = alpha_i * (T(x) - T_inf).
        # This is the tdep_props_on = False model of the training scripts,
        # i.e. the 'constant properties' cell of the paper's 2x2 factorial.
        # It does NOT depend on TD at all.
        Tdev = T_ip - T_inf
        for i in range(n_ph):
            kap = kap + kap_b[i] * w_p[:, i:i + 1]
            Eh = Eh + E_b[i] * w_p[:, i:i + 1]
            eps = eps + alp_b[i] * w[:, i:i + 1] * Tdev
    else:
        for i in range(n_ph):
            kap = kap + kap_b[i] * _poly_eval(cfg['kappa_T_coef'][i], dTp) * w_p[:, i:i + 1]
            Eh = Eh + E_b[i] * _poly_eval(cfg['E_T_coef'][i], dTp) * w_p[:, i:i + 1]
            eps = eps + alp_b[i] * (_poly_antideriv(cfg['alpha_T_coef'][i], dTp)
                                    - _poly_antideriv(cfg['alpha_T_coef'][i], dTp0)) \
                * w[:, i:i + 1]
    s_hat = (s_b * w_p).sum(dim=1)
    return kap, Eh, eps, s_hat, float(1.0 - inw.to(DT).mean())


# =====================================================================
# Thermal solve -- Picard on kappa(T); a single linear solve for 'const'
# =====================================================================
def solve_thermal(mesh, w, cfg, prop_eval, TD, solver, tol=1e-11, maxit=60):
    dev = mesh.dev
    th = cfg['thickness']
    conn, BT, Nsh, wdetJ = mesh.conn, mesh.BT, mesh.N, mesh.wdetJ
    n = mesh.Nn
    BtB = torch.einsum('kai,kaj->kij', BT, BT)              # [4int,4,4]
    f_shape = (Nsh * wdetJ[:, None]).sum(dim=0) * th        # [4]

    Tbc = torch.zeros(n, dtype=DT, device=dev)
    Tbc[mesh.fixed_T] = TD
    T = torch.full((n,), float(TD), dtype=DT, device=dev)

    it = 0
    for it in range(maxit):
        T_ip = T[conn] @ Nsh.T                              # [Ne, 4int]
        kap, _, _, s_hat, clampf = props(w, T_ip, cfg, prop_eval, TD)
        Ke = torch.einsum('ek,k,kij->eij', kap, wdetJ, BtB) * th
        vals = Ke.reshape(-1)
        f = torch.zeros(n, dtype=DT, device=dev)
        f.index_add_(0, conn.reshape(-1),
                     (s_hat[:, None] * f_shape[None, :]).reshape(-1))
        # rhs for the correction field T' = T - Tbc  (T'|_D = 0)
        KTbc = torch.zeros(n, dtype=DT, device=dev)
        KTbc.index_add_(0, mesh.rowsT, vals * Tbc[mesh.colsT])
        rhs = f - KTbc
        rhs = rhs.masked_fill(mesh.fixed_T, 0.0)
        dscale = float(Ke.diagonal(dim1=1, dim2=2).abs().mean())
        r, c, v = _apply_dirichlet(mesh.rowsT, mesh.colsT, vals, n,
                                   mesh.fixed_T, dscale)
        Tn = solver.solve(r, c, v, rhs, n, key=f'therm{id(mesh)}') + Tbc
        dn = float((Tn - T).abs().max())
        T = Tn
        if prop_eval == 'const':
            break                                            # linear
        if dn < tol * max(1.0, abs(TD)):
            break
    T_ip = T[conn] @ Nsh.T
    _, _, _, _, clampf = props(w, T_ip, cfg, prop_eval, TD)
    return T, T_ip, dict(n_picard=it + 1, picard_dT=dn, clamp_frac=clampf)


# =====================================================================
# Mechanical solve -- classic incremental Newton-Raphson
# =====================================================================
def solve_mech(mesh, w, T_ip, cfg, constitutive, prop_eval, TD, solver,
               tol=1e-11, maxit=30, n_inc=4, chunk=8192, verbose=False):
    dev = mesh.dev
    linelas = (constitutive == 'linelas')
    nu, th = cfg['nu'], cfg['thickness']
    K_out, uc = cfg['K_out'], cfg['uout_comp']
    Ne, n = mesh.Ne, 2 * mesh.Nn
    port_dof = 2 * mesh.port + uc

    _, Eh, eps_th, _, _ = props(w, T_ip, cfg, prop_eval, TD)
    c1 = Eh / (1 - nu ** 2)
    c2 = Eh / (2 * (1 + nu))
    BT_e = mesh.BT.unsqueeze(0).expand(Ne, 4, 2, 4)
    wdetJ_e = mesh.wdetJ.unsqueeze(0).expand(Ne, 4)
    # energy-interpolation factor gamma (exactly 0 / 1 on a binary map);
    # gamma is INERT under linelas (the Wang form collapses to psi_L)
    if linelas:
        g_e = torch.ones(Ne, 1, 1, 1, dtype=DT, device=dev)
    else:
        rho_s = w[:, 1:].sum(dim=1)
        bg, eg = cfg['gamma_beta'], cfg['gamma_eta']
        gam = (math.tanh(bg * eg) + torch.tanh(bg * (rho_s - eg))) / \
              (math.tanh(bg * eg) + math.tanh(bg * (1.0 - eg)))
        g_e = gam.clamp(0.0, 1.0).view(-1, 1, 1, 1)

    grad_e = torch.func.vmap(torch.func.grad(_elem_energy_dofs, argnums=0),
                             in_dims=(0, 0, 0, 0, 0, 0, 0, None, None, None))
    hess_e = torch.func.vmap(torch.func.hessian(_elem_energy_dofs, argnums=0),
                             in_dims=(0, 0, 0, 0, 0, 0, 0, None, None, None))

    def _chunked(fn, d_all, eth, out_shape):
        out = torch.empty((Ne,) + out_shape, dtype=DT, device=dev)
        step = chunk if chunk else Ne
        for a in range(0, Ne, step):
            b = min(a + step, Ne)
            out[a:b] = fn(d_all[a:b], BT_e[a:b], g_e[a:b], c1[a:b], c2[a:b],
                          eth[a:b], wdetJ_e[a:b], nu, th, linelas)
        return out

    def residual(u, eth):
        d_all = u[mesh.edof]
        ge = _chunked(grad_e, d_all, eth, (8,))
        R = torch.zeros(n, dtype=DT, device=dev)
        R.index_add_(0, mesh.edof.reshape(-1), ge.reshape(-1))
        R[port_dof] += K_out * u[port_dof]
        return R

    def tangent(u, eth):
        d_all = u[mesh.edof]
        he = _chunked(hess_e, d_all, eth, (8, 8))
        vals = he.reshape(-1)
        rows = torch.cat([mesh.rowsU,
                          torch.tensor([port_dof], device=dev, dtype=torch.long)])
        cols = torch.cat([mesh.colsU,
                          torch.tensor([port_dof], device=dev, dtype=torch.long)])
        vals = torch.cat([vals, torch.tensor([K_out], device=dev, dtype=DT)])
        dscale = float(he.diagonal(dim1=1, dim2=2).abs().mean())
        return rows, cols, vals, dscale

    u = torch.zeros(n, dtype=DT, device=dev)
    Rref = max(float(residual(u, eps_th).masked_fill(mesh.fixed_u, 0.0)
                     .abs().max()), 1e-300)
    n_newton, n_cut, converged = 0, 0, True

    if linelas:
        # the potential is QUADRATIC -> one linear solve is exact
        R = residual(u, eps_th).masked_fill(mesh.fixed_u, 0.0)
        rows, cols, vals, ds = tangent(u, eps_th)
        r, c, v = _apply_dirichlet(rows, cols, vals, n, mesh.fixed_u, ds)
        u = u + solver.solve(r, c, v, -R, n, key=f'mech{id(mesh)}')
        n_newton, n_inc_done = 1, 1
    else:
        s_done, ds_inc, n_inc_done = 0.0, 1.0 / n_inc, 0
        while s_done < 1.0 - 1e-12:
            s = min(s_done + ds_inc, 1.0)
            eth = eps_th * s
            u_try, ok, it = u.clone(), False, 0
            for it in range(maxit):
                R = residual(u_try, eth).masked_fill(mesh.fixed_u, 0.0)
                rn = float(R.abs().max())
                if rn < tol * Rref:
                    ok = True
                    break
                rows, cols, vals, dsc = tangent(u_try, eth)
                r, c, v = _apply_dirichlet(rows, cols, vals, n,
                                           mesh.fixed_u, dsc)
                du = solver.solve(r, c, v, -R, n, key=f'mech{id(mesh)}')
                if not bool(torch.isfinite(du).all()):
                    break
                u_try = u_try + du                     # full Newton step
                n_newton += 1
                if float(u_try.abs().max()) > 1e4:     # runaway guard
                    break
            if ok:
                u, s_done, n_inc_done = u_try, s, n_inc_done + 1
                if verbose:
                    print(f"    inc s={s:.4f}: {it} NR its, "
                          f"|R|/|R0|={rn / Rref:.2e}", flush=True)
            else:
                ds_inc *= 0.5
                n_cut += 1
                if ds_inc < 1e-4:
                    converged = False
                    break

    R = residual(u, eps_th).masked_fill(mesh.fixed_u, 0.0)
    rn = float(R.abs().max())
    U = u.view(-1, 2)
    H = torch.matmul(BT_e, u[mesh.edof].view(Ne, 1, 4, 2)).transpose(-1, -2)
    Habs = H.abs().amax(dim=(1, 2, 3))
    solid = w[:, 1:].sum(dim=1) > 0.5
    return U, dict(n_newton=n_newton, n_inc=n_inc_done, n_cut=n_cut,
                   converged=bool(converged), res_abs=rn, res_rel=rn / Rref,
                   Hmax=float(Habs.max()),
                   Hmax_solid=float(Habs[solid].max()) if bool(solid.any())
                   else float('nan'),
                   u_out=float(U[mesh.port, uc]),
                   dx_tip=float(U[mesh.port, 0]),
                   dy_tip=float(U[mesh.port, 1]))


def run_case(mesh, w, cfg, constitutive, prop_eval, TD, solver, **kw):
    T, T_ip, ti = solve_thermal(mesh, w, cfg, prop_eval, TD, solver)
    U, mi = solve_mech(mesh, w, T_ip, cfg, constitutive, prop_eval, TD,
                       solver, **kw)
    solid = w[:, 1:].sum(dim=1) > 0.5
    Te = T[mesh.conn].mean(dim=1)
    out = dict(constitutive=constitutive, prop_eval=prop_eval, TD=float(TD))
    out.update(ti); out.update(mi)
    out['T_min'] = float(T.min()); out['T_max'] = float(T.max())
    out['T_solid_min'] = float(Te[solid].min()) if bool(solid.any()) else float('nan')
    out['T_solid_max'] = float(Te[solid].max()) if bool(solid.any()) else float('nan')
    return out, T, U


def run_cross(mesh, w, cfg, TD, solver, n_inc=4, verbose=False, **kw):
    """Solve the SAME frozen design under all four evaluation cells.
    The thermal problem depends only on the property model (one-way
    coupling; solve_thermal never sees u), so each of the two temperature
    fields is solved ONCE and shared by both constitutive laws.
    Returns (res: {(const, pe): dict}, Ts: {pe: T}, Us: {(const, pe): U})."""
    solid = w[:, 1:].sum(dim=1) > 0.5
    res, Ts, Us = {}, {}, {}
    for pe in ('const', 'tdep'):
        T, T_ip, ti = solve_thermal(mesh, w, cfg, pe, TD, solver)
        Ts[pe] = T
        Te = T[mesh.conn].mean(dim=1)
        for const in ('linelas', 'hencky'):
            U, mi = solve_mech(mesh, w, T_ip, cfg, const, pe, TD, solver,
                               n_inc=n_inc, verbose=verbose, **kw)
            out = dict(eval_const=const, eval_prop=pe, TD=float(TD))
            out.update(ti); out.update(mi)
            out['T_min'] = float(T.min()); out['T_max'] = float(T.max())
            out['T_solid_min'] = (float(Te[solid].min()) if bool(solid.any())
                                  else float('nan'))
            out['T_solid_max'] = (float(Te[solid].max()) if bool(solid.any())
                                  else float('nan'))
            res[(const, pe)] = out
            Us[(const, pe)] = U
    return res, Ts, Us


# =====================================================================
# Design loading / folder-name parsing
# =====================================================================
def load_design(folder: Path, dev, cfg=CFG):
    """topo_material_id.csv -> (keep mask [Ngy,Ngx], one-hot weights, mat ids)."""
    d = np.loadtxt(Path(folder) / 'topo_material_id.csv', comments='%')
    Ngx, Ngy = cfg['unigrid']
    hx = (cfg['xmax'] - cfg['xmin']) / Ngx
    hy = (cfg['ymax'] - cfg['ymin']) / Ngy
    i = np.round((d[:, 0] - cfg['xmin'] - hx / 2) / hx).astype(int)
    j = np.round((d[:, 1] - cfg['ymin'] - hy / 2) / hy).astype(int)
    keep = np.zeros((Ngy, Ngx), dtype=bool)
    mat = np.full((Ngy, Ngx), -1, dtype=int)
    keep[j, i] = True
    mat[j, i] = np.round(d[:, 2]).astype(int)
    m = mat.reshape(-1)[keep.reshape(-1)]
    w = np.zeros((m.size, len(PHASE_NAMES)))
    w[np.arange(m.size), m] = 1.0
    return keep, torch.as_tensor(w, dtype=DT, device=dev), m


def parse_name(name: str):
    mt = re.match(r'Run_(\d+)_(linelas|hencky)_(\d+)$', name)
    if not mt:
        return None
    return dict(seed=int(mt.group(1)), constitutive=mt.group(2),
                TD=float(mt.group(3)))


# =====================================================================
# Verification (fast; run this before trusting any number)
# =====================================================================
def verify(dev=None, solver=None):
    dev = dev or ('cuda' if torch.cuda.is_available() else 'cpu')
    dev = torch.device(dev)
    solver = solver or SparseDirect()
    print("V1  free thermal expansion patch test "
          "(uniform material, uniform T, statically determinate BCs)")
    cfg = dict(CFG); cfg['unigrid'] = (6, 4)
    cfg['s'] = [0.0, 0.0, 0.0, 0.0]                # no source -> T == TD
    cfg['K_out'] = 0.0                             # no output spring
    m = Mesh(np.ones((4, 6), dtype=bool), dev, cfg)
    w = torch.zeros(m.Ne, 4, dtype=DT, device=dev); w[:, 2] = 1.0   # pure Cu
    TD = 673.0
    x, y = m.nodes[:, 0], m.nodes[:, 1]
    n00 = int(torch.argmin(x ** 2 + y ** 2))
    nx0 = int(torch.argmin((x - x.max()) ** 2 + y ** 2))
    fu = torch.zeros(2 * m.Nn, dtype=torch.bool, device=dev)
    fu[2 * n00] = fu[2 * n00 + 1] = fu[2 * nx0 + 1] = True
    m.fixed_u = fu
    for pe in ('const', 'tdep'):
        T, T_ip, _ = solve_thermal(m, w, cfg, pe, TD, solver)
        assert abs(float(T.max()) - TD) < 1e-9 and abs(float(T.min()) - TD) < 1e-9
        _, _, eps_th, _, _ = props(w, T_ip, cfg, pe, TD)
        e = float(eps_th.mean())
        assert float((eps_th - e).abs().max()) < 1e-12      # uniform field
        for const, fac in (('linelas', e), ('hencky', math.exp(e) - 1.0)):
            # linelas: eps = H - eps_th = 0 -> u = eps_th X
            # hencky : eps_H = log U = eps_th I -> u = (exp(eps_th) - 1) X
            U, _i = solve_mech(m, w, T_ip, cfg, const, pe, TD, solver, n_inc=4)
            err = float((U - fac * m.nodes).abs().max()
                        / (fac * m.nodes).abs().max())
            print(f"    {pe:5s} {const:8s} eps_th = {e:.6e}   "
                  f"rel err vs analytic = {err:.3e}")
            assert err < 1e-10, err

    print("V2  zero eigenstrain -> u == 0")
    cfg2 = dict(CFG); cfg2['unigrid'] = (6, 4)
    cfg2['alpha'] = [0.0, 0.0, 0.0, 0.0]
    m2 = Mesh(np.ones((4, 6), dtype=bool), dev, cfg2)
    w2 = torch.zeros(m2.Ne, 4, dtype=DT, device=dev); w2[:, 3] = 1.0
    T, T_ip, _ = solve_thermal(m2, w2, cfg2, 'tdep', 873.0, solver)
    for const in ('linelas', 'hencky'):
        U, _i = solve_mech(m2, w2, T_ip, cfg2, const, 'tdep', 873.0, solver, n_inc=2)
        print(f"    {const:8s} |u|max = {float(U.abs().max()):.3e}")
        assert float(U.abs().max()) < 1e-12

    print("V3  'const' invariants: == 'tdep' at T = T_ref exactly; "
          "T-invariant kappa/E; eps_th exactly linear in (T - T_inf)")
    Tref = cfg['prop_T_ref']
    a = props(w, T_ip * 0 + Tref, cfg, 'tdep', TD)
    b = props(w, T_ip * 0 + Tref, cfg, 'const', TD)
    for i, nm in enumerate(('kappa', 'E')):
        dd = float((a[i] - b[i]).abs().max() / a[i].abs().max().clamp_min(1e-30))
        print(f"    {nm:7s} tdep vs const at T_ref:      max rel diff = {dd:.3e}")
        assert dd < 1e-12
    assert float(a[2].abs().max()) == 0.0 and float(b[2].abs().max()) == 0.0
    print(f"    eps_th  both exactly 0 at T_ref:  OK")
    c1 = props(w, T_ip * 0 + 600.0, cfg, 'const', TD)
    c2 = props(w, T_ip * 0 + 1000.0, cfg, 'const', TD)
    for i, nm in enumerate(('kappa', 'E')):
        dd = float((c1[i] - c2[i]).abs().max())
        print(f"    {nm:7s} const, 600 K vs 1000 K:      max abs diff = {dd:.3e}")
        assert dd == 0.0
    r1 = c1[2] / (600.0 - cfg['T_inf'])
    r2 = c2[2] / (1000.0 - cfg['T_inf'])
    dd = float((r1 - r2).abs().max() / r1.abs().max().clamp_min(1e-30))
    print(f"    eps_th/(T - T_inf), 600 vs 1000 K: max rel diff = {dd:.3e}")
    assert dd < 1e-12
    print("VERIFICATION PASSED\n")


def check_verbatim(train_script=None):
    """Assert the copied constitutive block is CODE-identical to the training
    script's.  Compares normalised ASTs with docstrings stripped, so only the
    executable statements are compared (the docstrings here are abridged).
    Runs on the first training file found next to this one (pass a path to
    pin it); skipped if none is present."""
    import ast, glob as _glob, inspect, sys
    cands = ([train_script] if train_script else
             sorted(_glob.glob('run_actuator.py')) + ['ex1_jul27.py'] + sorted(_glob.glob('EX2D[12]_*.py')))
    p = next((Path(c) for c in cands if c and Path(c).exists()), None)
    if p is None:
        print("[verbatim] no training script found -- skipping identity check")
        return None
    names = ['_f_atanh', '_hencky_eps_from_C', 'hencky_strain_2d',
             'W_density_hencky_thermal', 'W_density_linear_thermal',
             '_poly_eval', '_poly_antideriv', '_elem_energy_dofs']

    def sig(fn_node):
        fn_node = ast.parse(ast.unparse(fn_node)).body[0]
        if (fn_node.body and isinstance(fn_node.body[0], ast.Expr)
                and isinstance(fn_node.body[0].value, ast.Constant)
                and isinstance(fn_node.body[0].value.value, str)):
            fn_node.body = fn_node.body[1:]          # drop the docstring
        return ast.dump(fn_node)

    ref = {n.name: sig(n) for n in ast.parse(p.read_text()).body
           if isinstance(n, ast.FunctionDef) and n.name in names}
    here = sys.modules[__name__]
    bad, missing = [], [n for n in names if n not in ref]
    for n in names:
        if n in ref:
            mine = ast.parse(inspect.getsource(getattr(here, n))).body[0]
            if sig(mine) != ref[n]:
                bad.append(n)
    ok = not bad and not missing
    print(f"[verbatim] {len(names)} constitutive functions vs {train_script}: "
          + ("CODE-IDENTICAL" if ok
             else f"DIFFER -> {bad}" + (f"  MISSING -> {missing}" if missing else "")))
    return bad + missing


# =====================================================================
# Cross-evaluation sweep -- every design x eval TDs x the four cells
# =====================================================================
def _matrix(rows):
    """Fold the OWN-TD rows (evaluation TD == training TD) into one record
    per design: the four u_out values plus the factorial effects, all
    normalised by the REFERENCE cell u_ref = u(hencky, tdep) -- the
    highest-fidelity model in the study -- so the effects share one
    denominator and are directly comparable."""
    by = {}
    for r in rows:
        if float(r['TD']) != float(r['TD_design']):
            continue
        by.setdefault(r['folder'], {})[(r['eval_const'], r['eval_prop'])] = r
    recs = []
    for folder, cells in by.items():
        if len(cells) != 4:
            continue
        a = next(iter(cells.values()))
        uLc = cells[('linelas', 'const')]['u_out']
        uLt = cells[('linelas', 'tdep')]['u_out']
        uHc = cells[('hencky', 'const')]['u_out']
        uHt = cells[('hencky', 'tdep')]['u_out']
        ref = uHt
        dconst = a['design_const']
        u_nat = cells[(dconst, PROP_FOR_CONST[dconst])]['u_out']
        recs.append(dict(
            folder=folder, seed=a['seed'], design_const=dconst, TD=a['TD'],
            u_linelas_const=uLc, u_linelas_tdep=uLt,
            u_hencky_const=uHc, u_hencky_tdep=uHt,
            u_native=u_nat, u_ref=ref,
            # constitutive effect (hencky - linelas) at each property model
            const_eff_constp=(uHc - uLc) / ref,
            const_eff_tdep=(uHt - uLt) / ref,
            # property effect (tdep - const) under each constitutive law
            prop_eff_linelas=(uLt - uLc) / ref,
            prop_eff_hencky=(uHt - uHc) / ref,
            # 2x2 interaction on the same scale
            interaction=(uHt - uHc - uLt + uLc) / ref,
            # how far the design's own training physics mis-states u_ref
            native_vs_ref=(u_nat - ref) / ref))
    recs.sort(key=lambda r: (r['TD'], r['design_const'], r['seed']))
    return recs


def _robust(rows, cell=None):
    """One record per design: u_out under the judging cell (default
    REF_CELL = hencky + tdep) at every evaluation TD, plus the end-to-end
    span and the mid-range curvature."""
    const, pe = cell or REF_CELL
    by = {}
    for r in rows:
        if r['eval_const'] == const and r['eval_prop'] == pe:
            by.setdefault(r['folder'], {})[float(r['TD'])] = r
    recs = []
    for folder, d in by.items():
        tds = sorted(d)
        a = d[tds[0]]
        rec = dict(folder=folder, seed=a['seed'],
                   design_const=a['design_const'],
                   TD_design=float(a['TD_design']))
        for td in tds:
            rec[f'u_ref_{int(td)}'] = d[td]['u_out']
        if len(tds) >= 2:
            rec['span'] = d[tds[-1]]['u_out'] - d[tds[0]]['u_out']
        if len(tds) >= 3:
            mid = tds[len(tds) // 2]
            rec['curvature'] = (d[mid]['u_out']
                                - 0.5 * (d[tds[0]]['u_out'] + d[tds[-1]]['u_out']))
        recs.append(rec)
    recs.sort(key=lambda r: (r['TD_design'], r['design_const'], r['seed']))
    return recs


def sweep(root='processed_July28', out='results', device=None, backend='auto',
          n_inc=4, verbose=False, save_fields='native', eval_tds=None):
    """CROSS-EVALUATION + ROBUSTNESS sweep.  Every design folder is solved
    in all four cells of EVAL_CELLS ({linelas, hencky} x {const, tdep}) at
    every TD in eval_tds (default EVAL_TDS; own TD always included).
    Writes  out/cross_eval.csv     -- long, one row per solve
            out/cross_matrix.csv   -- wide, one row per design (own-TD 2x2)
            out/robust_refcell.csv -- wide, u_out vs eval TD under REF_CELL
    save_fields: False | 'native' (fields at the design's own TD only)
                 | True (fields at every eval TD, ~len(eval_tds)x files)."""
    eval_tds = [float(t) for t in (EVAL_TDS if eval_tds is None else eval_tds)]
    dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    print(f"[device] {dev}"
          + (f"  ({torch.cuda.get_device_name(0)})" if dev.type == 'cuda' else ""))
    solver = SparseDirect(backend=backend)
    root, outdir = Path(root), Path(out)
    outdir.mkdir(parents=True, exist_ok=True)
    folders = sorted([p for p in root.iterdir() if p.is_dir()
                      and (p / 'topo_material_id.csv').exists()],
                     key=lambda p: (float(p.name.split('_')[-1]),
                                    p.name.split('_')[2],
                                    int(p.name.split('_')[1])))
    metas = [(f, parse_name(f.name)) for f in folders]
    for f, m in metas:
        if m is None:
            print(f"[skip] {f.name}: unparsable folder name")
    metas = [(f, m) for f, m in metas if m is not None]
    n_solves = sum(len(set(eval_tds) | {m['TD']}) * len(EVAL_CELLS)
                   for _, m in metas)
    print(f"{len(metas)} designs x eval TDs {sorted(set(eval_tds))} "
          f"x {len(EVAL_CELLS)} cells = {n_solves} solves\n")
    rows, mesh_cache, t0 = [], {}, time.time()
    for k, (f, meta) in enumerate(metas, 1):
        dconst, TDd = meta['constitutive'], meta['TD']
        native_pe = PROP_FOR_CONST[dconst]
        keep, w, m = load_design(f, dev)
        key = keep.tobytes()
        if key not in mesh_cache:
            mesh_cache[key] = Mesh(keep, dev)
        mesh = mesh_cache[key]
        fr = {f'frac_{PHASE_NAMES[i]}': float((m == i).mean())
              for i in range(4)}
        for td in sorted(set(eval_tds) | {TDd}):
            ts = time.time()
            res, Ts, Us = run_cross(mesh, w, CFG, td, solver,
                                    n_inc=n_inc, verbose=verbose)
            wall = round(time.time() - ts, 2)
            for (const, pe) in EVAL_CELLS:
                rows.append(dict(folder=f.name, seed=meta['seed'],
                                 design_const=dconst, TD_design=TDd,
                                 native=int(const == dconst and pe == native_pe
                                            and td == TDd),
                                 **res[(const, pe)], **fr, wall_s=wall))
            u = {c: res[c]['u_out'] for c in EVAL_CELLS}
            worst = max(res[c]['res_rel'] for c in EVAL_CELLS)
            own = '*' if td == TDd else ' '
            print(f"[{k:2d}/{len(metas)}] {f.name:22s} eval TD={td:5.0f}{own} "
                  f"u_out: L/const={u[('linelas','const')]:+8.4f}  "
                  f"L/tdep={u[('linelas','tdep')]:+8.4f}  "
                  f"H/const={u[('hencky','const')]:+8.4f}  "
                  f"H/tdep={u[('hencky','tdep')]:+8.4f}   "
                  f"|R|/|R0|<{worst:.0e}  ({wall}s)", flush=True)
            if save_fields is True or (save_fields == 'native' and td == TDd):
                sfx = '' if td == TDd else f'_evalTD{int(td)}'
                np.savez_compressed(
                    outdir / f'{f.name}{sfx}_fields.npz',
                    nodes=mesh.nodes.cpu().numpy(),
                    conn=mesh.conn.cpu().numpy(), mat=m,
                    **{f'T_{pe}': Ts[pe].cpu().numpy() for pe in Ts},
                    **{f'U_{cell_tag(c, pe)}': Us[(c, pe)].cpu().numpy()
                       for (c, pe) in Us})
    order = ['folder', 'seed', 'design_const', 'TD_design', 'TD', 'native',
             'eval_const', 'eval_prop', 'u_out', 'dx_tip', 'dy_tip', 'Hmax',
             'Hmax_solid', 'T_min', 'T_max', 'T_solid_min', 'T_solid_max',
             'clamp_frac', 'n_picard', 'picard_dT', 'n_newton', 'n_inc',
             'n_cut', 'converged', 'res_abs', 'res_rel', 'wall_s']
    keys = order + sorted({k for r in rows for k in r} - set(order))
    with open(outdir / 'cross_eval.csv', 'w', newline='') as fh:
        wr = csv.DictWriter(fh, fieldnames=keys); wr.writeheader()
        for r in rows:
            wr.writerow(r)
    recs = _matrix(rows)
    if recs:
        with open(outdir / 'cross_matrix.csv', 'w', newline='') as fh:
            wr = csv.DictWriter(fh, fieldnames=list(recs[0])); wr.writeheader()
            for r in recs:
                wr.writerow(r)
    rob = _robust(rows)
    if rob:
        rkeys = sorted({k for r in rob for k in r},
                       key=lambda k: (k not in ('folder', 'seed',
                                                'design_const', 'TD_design'), k))
        with open(outdir / 'robust_refcell.csv', 'w', newline='') as fh:
            wr = csv.DictWriter(fh, fieldnames=rkeys); wr.writeheader()
            for r in rob:
                wr.writerow(r)
    print(f"\nwrote {outdir / 'cross_eval.csv'}, "
          f"{outdir / 'cross_matrix.csv'} and "
          f"{outdir / 'robust_refcell.csv'}  "
          f"({len(rows)} solves in {time.time() - t0:.1f}s)")
    return rows


# =====================================================================
# Statistics -- own-TD 2x2 tables (A-E) + temperature robustness (F-G)
# =====================================================================
def report(rows, out='results'):
    import numpy as np
    recs = _matrix(rows)
    rob = _robust(rows)
    all_tds = sorted({float(r['TD']) for r in rows
                      if (r['eval_const'], r['eval_prop']) == REF_CELL})
    L = []
    P = L.append
    TDs = sorted({r['TD'] for r in recs})
    P("=" * 112)
    P(f"{EX_TITLE.get(CFG.get('example'), CFG.get('example'))} -- "
      f"2x2 CROSS-EVALUATION + TEMPERATURE ROBUSTNESS")
    P("=" * 112)
    comp = 'vertical (+v)' if CFG['uout_comp'] == 1 else 'horizontal (+u)'
    P(f"u_out = {comp} stroke (um) at the output-port node "
      f"{tuple(CFG['port_xy'])}")
    P(f"{len(recs)} designs;  A-E: physics cross at each design's OWN TD;  "
      f"F-G: every design evaluated at TD in "
      f"{{{', '.join(f'{t:.0f}' for t in all_tds)}}} K")
    P(f"reference cell for all % effects and for the robustness judge: "
      f"u_ref = u(hencky, T-dep);  'const' = room-temperature base "
      f"properties, eps_th = alpha_base*(T - T_inf)")
    P("")
    P(f"solver audit (all {len(rows)} solves): non-converged = "
      f"{sum(1 for r in rows if not r['converged'])};  "
      f"max |R|/|R0| = {max(r['res_rel'] for r in rows):.2e};  "
      f"max clamp frac = {max(r['clamp_frac'] for r in rows):.3f};  "
      f"max |grad u| (solid) = {max(r['Hmax_solid'] for r in rows):.3f} (cap 1.0)")
    P("")

    # ---------------- A. per design: the raw 2x2 matrix (own TD) ----------
    P("-" * 112)
    P("A.  per design at its OWN TD -- u_out (um) in the four evaluation "
      "cells (* marks the design's native/training cell)")
    P("-" * 112)
    P(f"{'design':<24s}{'TD':>6s} | {'lin+const':>9s} {'lin+tdep':>9s} "
      f"{'hen+const':>9s} {'hen+tdep':>9s} | {'dCONST':>8s} {'dPROPS':>8s} "
      f"{'nat-ref':>8s}")
    P(f"{'':<24s}{'':>6s} | {'':>9s} {'':>9s} {'':>9s} {'(=ref)':>9s} | "
      f"{'@tdep':>8s} {'@hencky':>8s} {'':>8s}")
    for r in recs:
        nat = (r['design_const'], PROP_FOR_CONST[r['design_const']])
        def fmt(c, v):
            return f"{v:>8.4f}" + ("*" if c == nat else " ")
        P(f"{r['folder']:<24s}{r['TD']:>6.0f} | "
          f"{fmt(('linelas','const'), r['u_linelas_const'])} "
          f"{fmt(('linelas','tdep'),  r['u_linelas_tdep'])} "
          f"{fmt(('hencky','const'),  r['u_hencky_const'])} "
          f"{fmt(('hencky','tdep'),   r['u_hencky_tdep'])} | "
          f"{100 * r['const_eff_tdep']:>7.2f}% "
          f"{100 * r['prop_eff_hencky']:>7.2f}% "
          f"{100 * r['native_vs_ref']:>7.2f}%")
    P("")

    # ---------------- B. cell means by TD x design family -----------------
    P("-" * 112)
    P("B.  mean u_out (um) over seeds at the OWN TD, by TD x design family "
      "x evaluation cell")
    P("-" * 112)
    P(f"{'group':<30s}{'n':>3s} | {'lin+const':>9s} {'lin+tdep':>9s} "
      f"{'hen+const':>9s} {'hen+tdep':>9s}")
    keymap = [('u_linelas_const', ('linelas', 'const')),
              ('u_linelas_tdep', ('linelas', 'tdep')),
              ('u_hencky_const', ('hencky', 'const')),
              ('u_hencky_tdep', ('hencky', 'tdep'))]
    for td in TDs:
        for dc in ('linelas', 'hencky'):
            g = [r for r in recs if r['TD'] == td and r['design_const'] == dc]
            if not g:
                continue
            mu = [np.mean([r[k] for r in g]) for k, _ in keymap]
            P(f"TD={td:5.0f} K  {dc + '-designed':<17s}{len(g):>3d} | "
              + " ".join(f"{m:>9.4f}" for m in mu))
    P("")

    # ---------------- C. factorial effects ------------------------------
    P("-" * 112)
    P("C.  factorial effects at the OWN TD, % of u_ref  (mean +/- sd over "
      "designs at each TD)")
    P("    dCONST = hencky - linelas at fixed property model;  "
      "dPROPS = tdep - const at fixed constitutive law")
    P("-" * 112)
    P(f"{'group':<26s}{'n':>3s} {'dCONST@conp':>12s} {'dCONST@tdep':>12s} "
      f"{'dPROPS@lin':>12s} {'dPROPS@hen':>12s} {'interaction':>12s}")

    def eff_line(g, name):
        def ms(k):
            v = np.array([r[k] for r in g])
            sd = v.std(ddof=1) if len(v) > 1 else 0.0
            return f"{100 * v.mean():>6.2f}+/-{100 * sd:<4.2f}"
        P(f"{name:<26s}{len(g):>3d} {ms('const_eff_constp'):>12s} "
          f"{ms('const_eff_tdep'):>12s} {ms('prop_eff_linelas'):>12s} "
          f"{ms('prop_eff_hencky'):>12s} {ms('interaction'):>12s}")

    for td in TDs:
        eff_line([r for r in recs if r['TD'] == td], f"TD = {td:.0f} K")
    eff_line(recs, "ALL")
    P("")

    # ---------------- D. design-family comparison under each judge --------
    P("-" * 112)
    P("D.  which design family wins under each evaluation physics "
      "('judge'), at the OWN TD?  gap = (hencky-designed - "
      "linelas-designed) / hencky-designed, on mean u_out")
    P("-" * 112)
    P(f"{'judge (evaluation cell)':<28s}" +
      "".join(f"{'TD=' + format(td, '.0f'):>16s}{'gap':>9s}" for td in TDs))
    for k, cell in keymap:
        line = f"{cell[0] + ' + ' + cell[1]:<28s}"
        for td in TDs:
            gl = [r[k] for r in recs
                  if r['TD'] == td and r['design_const'] == 'linelas']
            gh = [r[k] for r in recs
                  if r['TD'] == td and r['design_const'] == 'hencky']
            if gl and gh:
                mL, mH = np.mean(gl), np.mean(gh)
                line += f"{mH:>7.3f}/{mL:<7.3f}{100 * (mH - mL) / mH:>8.2f}%"
            else:
                line += f"{'--':>16s}{'':>9s}"
        P(line)
    P("    (each TD column shows: mean u_out hencky-designed / "
      "linelas-designed under that judge)")
    P("")

    # ---------------- E. self-assessment bias ---------------------------
    P("-" * 112)
    P("E.  self-assessment bias at the OWN TD: u at the design's NATIVE "
      "cell (the factorial pairing its family represents) vs the "
      "reference cell u(hencky, tdep)")
    P("-" * 112)
    for td in TDs:
        for dc in ('linelas', 'hencky'):
            g = [r for r in recs if r['TD'] == td and r['design_const'] == dc]
            if not g:
                continue
            v = 100 * np.array([r['native_vs_ref'] for r in g])
            sd = v.std(ddof=1) if len(v) > 1 else 0.0
            nat = f"{dc}+{PROP_FOR_CONST[dc]}"
            note = ("   (native = reference cell)"
                    if (dc, PROP_FOR_CONST[dc]) == ('hencky', 'tdep') else "")
            P(f"TD={td:5.0f} K  {dc + '-designed':<17s} "
              f"(u_native - u_ref)/u_ref [{nat:<16s}] = "
              f"{v.mean():>+7.2f}% +/- {sd:.2f}%{note}")
    P("")

    # ---------------- F. temperature robustness per design ---------------
    P("-" * 112)
    P("F.  temperature robustness -- every design judged under the "
      "reference physics (hencky + T-dep props) at each evaluation TD")
    P("    (* marks the design's TRAINING TD;  span = u(TDmax) - u(TDmin);"
      "  curv = u(mid) - mean(u(ends)))")
    P("-" * 112)
    hdr = f"{'design':<24s}{'trained':>8s} |"
    for td in all_tds:
        hdr += f" {'u@' + format(td, '.0f'):>10s}"
    hdr += f" | {'span':>8s} {'curv':>8s}"
    P(hdr)
    for r in rob:
        line = f"{r['folder']:<24s}{r['TD_design']:>7.0f}K |"
        for td in all_tds:
            v = r.get(f'u_ref_{int(td)}')
            mark = '*' if td == r['TD_design'] else ' '
            line += f" {v:>9.4f}{mark}" if v is not None else f" {'--':>10s}"
        line += (f" | {r.get('span', float('nan')):>8.4f}"
                 f" {r.get('curvature', float('nan')):>8.4f}")
        P(line)
    P("")

    # ---------------- G. transferability across training TDs --------------
    P("-" * 112)
    P("G.  transferability: at each OPERATING TD, all designs judged under "
      "hencky + T-dep props at that TD;")
    P("    family = constitutive it was designed with x TD it was designed "
      "for;  gap = to the best family mean at that operating TD")
    P("-" * 112)
    fams = sorted({(r['design_const'], r['TD_design']) for r in rob},
                  key=lambda t: (t[1], t[0]))
    for td in all_tds:
        stats = []
        for dc, tdd in fams:
            v = np.array([r[f'u_ref_{int(td)}'] for r in rob
                          if r['design_const'] == dc and r['TD_design'] == tdd
                          and f'u_ref_{int(td)}' in r])
            if v.size:
                stats.append((dc, tdd, v.mean(),
                              v.std(ddof=1) if v.size > 1 else 0.0, v.size))
        if not stats:
            continue
        best = max(s[2] for s in stats)
        P(f"operating TD = {td:.0f} K:")
        for dc, tdd, mu, sd, nn in sorted(stats, key=lambda s: -s[2]):
            gap = 100 * (best - mu) / best
            tag = '  <-- best' if mu == best else f'   gap {gap:5.2f}%'
            P(f"    {dc + '-designed':<17s} @ {tdd:5.0f} K   n={nn}  "
              f"mean = {mu:8.4f} +/- {sd:6.4f} um{tag}")
        P("")
    P("=" * 112)
    txt = "\n".join(L)
    print(txt)
    Path(out, 'cross_stats.txt').write_text(txt + "\n")
    return txt


def _house_style():
    import matplotlib as mpl
    mpl.use('Agg')
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Computer Modern Roman"],
        "mathtext.fontset": "cm", "font.size": 16, "axes.linewidth": 1.0,
        "xtick.direction": "out", "ytick.direction": "out",
        "xtick.major.size": 4, "ytick.major.size": 4,
        "axes.formatter.use_mathtext": True})


def _light(hex_, f=0.55):
    """Mix a hex colour toward white by fraction f."""
    h = hex_.lstrip('#')
    r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
    return (r + (1 - r) * f, g + (1 - g) * f, b + (1 - b) * f)


COL = {'linelas': '#8da4b8', 'hencky': '#b87333'}
TD_COL = {673.0: '#8da4b8', 873.0: '#c2a15a', 1073.0: '#b0552f'}


def _td_col(td):
    return TD_COL.get(float(td), '#888888')


def _cell_face(const, pe):
    return COL[const] if pe == 'tdep' else _light(COL[const])


def figure(rows, out='results'):
    """Fig 1: per-design 2x2 cross at the OWN TD (native cell marked).
    Fig 2: seed-averaged 2x2 matrix per TD x design family (+/- sd).
    Fig 3: per-design u_out vs evaluation TD under the reference cell
    (training TD marked).  Fig 4: family-mean u_out vs evaluation TD."""
    _house_style()
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    recs = _matrix(rows)
    rob = _robust(rows)
    TDs = sorted({r['TD'] for r in recs})
    all_tds = sorted({float(r['TD']) for r in rows
                      if (r['eval_const'], r['eval_prop']) == REF_CELL})
    keymap = [('u_linelas_const', ('linelas', 'const')),
              ('u_linelas_tdep', ('linelas', 'tdep')),
              ('u_hencky_const', ('hencky', 'const')),
              ('u_hencky_tdep', ('hencky', 'tdep'))]
    handles = [Patch(facecolor=_cell_face(c, p), edgecolor='k',
                     hatch='///' if p == 'const' else None,
                     label=cell_label(c, p)) for _, (c, p) in keymap]

    # ---------------- figure 1: per design, own-TD cross -----------------
    fig, ax = plt.subplots(figsize=(14.56, 4.8))
    fig.subplots_adjust(left=0.06, right=0.995, top=0.93, bottom=0.30)
    xs = np.arange(len(recs))
    off = {0: -0.30, 1: -0.10, 2: 0.10, 3: 0.30}
    for i, (k, (c, p)) in enumerate(keymap):
        ax.bar(xs + off[i], [r[k] for r in recs], width=0.19,
               color=_cell_face(c, p), edgecolor='k', linewidth=0.35,
               hatch='///' if p == 'const' else None)
    for j, r in enumerate(recs):        # mark the native cell
        i = [q for q, (k, cell) in enumerate(keymap)
             if cell == (r['design_const'], PROP_FOR_CONST[r['design_const']])][0]
        ax.plot(j + off[i], r[[k for k, _ in keymap][i]] * 1.015 + 0.15,
                marker='v', ms=4, color='k', clip_on=False)
    for i in range(1, len(recs)):
        if recs[i]['TD'] != recs[i - 1]['TD']:
            ax.axvline(i - 0.5, color='0.4', lw=0.9, ls='--')
    ax.set_xticks(xs)
    ax.set_xticklabels([r['folder'].replace('Run_', '') for r in recs],
                       rotation=90, fontsize=11)
    ax.set_ylabel(r'$u_\mathrm{out}$ ($\mu$m)')
    ax.set_xlim(-0.7, len(recs) - 0.3)
    ax.legend(handles=handles + [plt.Line2D([], [], marker='v', ls='none',
                                            ms=5, color='k',
                                            label='native (training) cell')],
              fontsize=11, loc='upper left', framealpha=0.95, ncol=2)
    for td in TDs:
        idx = [i for i, r in enumerate(recs) if r['TD'] == td]
        ax.text(np.mean(idx), ax.get_ylim()[1] * 0.98, f'$T_D$ = {td:.0f} K',
                ha='center', va='top', fontsize=14)
    fig.savefig(Path(out, 'fig_cross_uout.png'), dpi=600)
    plt.close(fig)
    print(f"wrote {Path(out, 'fig_cross_uout.png')}")

    # ---------------- figure 2: seed-averaged own-TD matrix --------------
    groups = [(td, dc) for td in TDs for dc in ('linelas', 'hencky')
              if any(r['TD'] == td and r['design_const'] == dc for r in recs)]
    fig, ax = plt.subplots(figsize=(14.56, 4.0))
    fig.subplots_adjust(left=0.06, right=0.995, top=0.90, bottom=0.17)
    xg = np.arange(len(groups))
    for i, (k, (c, p)) in enumerate(keymap):
        mu, sd = [], []
        for td, dc in groups:
            v = np.array([r[k] for r in recs
                          if r['TD'] == td and r['design_const'] == dc])
            mu.append(v.mean())
            sd.append(v.std(ddof=1) if len(v) > 1 else 0.0)
        ax.bar(xg + off[i], mu, width=0.19, yerr=sd, capsize=2.5,
               error_kw=dict(lw=0.9), color=_cell_face(c, p), edgecolor='k',
               linewidth=0.5, hatch='///' if p == 'const' else None)
    for i in range(1, len(groups)):
        if groups[i][0] != groups[i - 1][0]:
            ax.axvline(i - 0.5, color='0.4', lw=0.9, ls='--')
    ax.set_xticks(xg)
    ax.set_xticklabels([f'$T_D$={td:.0f} K\n{dc}-designed'
                        for td, dc in groups], fontsize=13)
    ax.set_ylabel(r'$\overline{u}_\mathrm{out}$ ($\mu$m)')
    ax.set_xlim(-0.7, len(groups) - 0.3)
    ax.legend(handles=handles, fontsize=11, loc='upper left',
              framealpha=0.95, ncol=2)
    fig.savefig(Path(out, 'fig_cross_matrix.png'), dpi=600)
    plt.close(fig)
    print(f"wrote {Path(out, 'fig_cross_matrix.png')}")

    if len(all_tds) < 2 or not rob:
        return

    # ---------------- figure 3: per-design robustness bars ---------------
    nt = len(all_tds)
    wdt = 0.78 / nt
    offs = [(-0.39 + wdt / 2) + i * wdt for i in range(nt)]
    fig, ax = plt.subplots(figsize=(14.56, 4.8))
    fig.subplots_adjust(left=0.06, right=0.995, top=0.93, bottom=0.30)
    xs = np.arange(len(rob))
    for i, td in enumerate(all_tds):
        ax.bar(xs + offs[i],
               [r.get(f'u_ref_{int(td)}', np.nan) for r in rob], width=wdt,
               color=_td_col(td), edgecolor='k', linewidth=0.35,
               label=f'eval $T_D$ = {td:.0f} K')
    for j, r in enumerate(rob):         # mark the training TD
        if r['TD_design'] in all_tds:
            i = all_tds.index(r['TD_design'])
            ax.plot(j + offs[i],
                    r[f'u_ref_{int(r["TD_design"])}'] * 1.015 + 0.15,
                    marker='v', ms=4, color='k', clip_on=False)
    for i in range(1, len(rob)):
        if rob[i]['TD_design'] != rob[i - 1]['TD_design']:
            ax.axvline(i - 0.5, color='0.4', lw=0.9, ls='--')
    ax.set_xticks(xs)
    ax.set_xticklabels([r['folder'].replace('Run_', '') for r in rob],
                       rotation=90, fontsize=11)
    ax.set_ylabel(r'$u_\mathrm{out}$ ($\mu$m)')
    ax.set_xlim(-0.7, len(rob) - 0.3)
    ax.legend(handles=(ax.get_legend_handles_labels()[0]
                       + [plt.Line2D([], [], marker='v', ls='none', ms=5,
                                     color='k', label='training $T_D$')]),
              fontsize=11, loc='upper left', framealpha=0.95, ncol=2)
    for tdd in sorted({r['TD_design'] for r in rob}):
        idx = [i for i, r in enumerate(rob) if r['TD_design'] == tdd]
        ax.text(np.mean(idx), ax.get_ylim()[1] * 0.98,
                f'trained @ {tdd:.0f} K', ha='center', va='top', fontsize=14)
    fig.savefig(Path(out, 'fig_robust_uout.png'), dpi=600)
    plt.close(fig)
    print(f"wrote {Path(out, 'fig_robust_uout.png')}")

    # ---------------- figure 4: family means vs evaluation TD ------------
    fams = sorted({(r['design_const'], r['TD_design']) for r in rob},
                  key=lambda t: (t[1], t[0]))
    fig, ax = plt.subplots(figsize=(14.56, 4.8))
    fig.subplots_adjust(left=0.06, right=0.995, top=0.95, bottom=0.14)
    for dc, tdd in fams:
        mu, sd = [], []
        for td in all_tds:
            v = np.array([r[f'u_ref_{int(td)}'] for r in rob
                          if r['design_const'] == dc and r['TD_design'] == tdd
                          and f'u_ref_{int(td)}' in r])
            mu.append(v.mean() if v.size else np.nan)
            sd.append(v.std(ddof=1) if v.size > 1 else 0.0)
        ls = '-' if dc == 'hencky' else '--'
        mk = 'o' if dc == 'hencky' else 's'
        ax.errorbar(all_tds, mu, yerr=sd, color=_td_col(tdd), ls=ls,
                    marker=mk, ms=6, lw=1.6, capsize=3,
                    label=f'{dc}-designed @ {tdd:.0f} K')
        if tdd in all_tds:              # emphasise the on-design point
            i = all_tds.index(tdd)
            ax.plot(tdd, mu[i], marker=mk, ms=10, mfc='none', mec='k',
                    mew=1.4, ls='none', zorder=5)
    ax.set_xticks(all_tds)
    ax.set_xticklabels([f'{t:.0f}' for t in all_tds])
    ax.set_xlabel(r'evaluation $T_D$ (K)')
    ax.set_ylabel(r'$\overline{u}_\mathrm{out}$ ($\mu$m)')
    ax.legend(fontsize=11, loc='upper left', framealpha=0.95, ncol=2)
    fig.savefig(Path(out, 'fig_robust_lines.png'), dpi=600)
    plt.close(fig)
    print(f"wrote {Path(out, 'fig_robust_lines.png')}")


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--root', default='processed_July28')
    ap.add_argument('--out', default='results')
    ap.add_argument('--device', default=None)
    ap.add_argument('--backend', default='auto',
                    choices=['auto', 'torch_sla', 'cupy', 'scipy'])
    ap.add_argument('--n-inc', type=int, default=4,
                    help='thermal load increments for the hencky NR path')
    ap.add_argument('--no-verify', action='store_true')
    ap.add_argument('--save-fields', action='store_true')
    ap.add_argument('--verbose', action='store_true')
    a = ap.parse_args()
    check_verbatim()
    if not a.no_verify:
        verify(a.device)
    rows = sweep(a.root, a.out, device=a.device, backend=a.backend,
                 n_inc=a.n_inc, verbose=a.verbose, save_fields=a.save_fields)
    report(rows, a.out)
    figure(rows, a.out)


## 4 · Which backend actually works here

Reports the probe result. If it lands on `scipy` you still get correct numbers — expect a few minutes for the whole sweep instead of well under one.

In [ ]:
import importlib, ex2d2_forward as fw
importlib.reload(fw)

solver = fw.SparseDirect()          # probes torch_sla -> cupy -> scipy
print("\nselected backend:", solver.backend)
if solver.backend != 'torch_sla':
    print("cuDSS unavailable -- falling back. Optional things to try:\n"
          "  !pip install -q -U torch-sla nvmath-python\n"
          "  or pin an older nvmath: !pip install -q 'nvmath-python[cu12]<0.6'\n"
          "then re-run this cell. Results are identical either way.")

## 5 · Verification

Run this before trusting any number.

- **V1 free-expansion patch test**, now under **both property models** — uniform material, uniform
  `T`, statically determinate BCs. `linelas` must give `u = ε_th·X` exactly; `hencky` must give
  `u = (e^{ε_th} − 1)·X` exactly (isotropic log-strain). Under `const`, `ε_th = α_Cu·(T_D − T_inf)`
  exactly; under `tdep` it is the exact CTE integral. The four analytic answers differ, so the test
  tells all four paths apart.
- **V2** zero CTE → `u ≡ 0`.
- **V3** `const` invariants: equals `tdep` **exactly at `T = T_ref = 293 K`** (the f-polynomials
  are normalised to 1 there); `κ`/`E` bit-identical at any two temperatures (true `T`-invariance);
  `ε_th` exactly linear in `(T − T_inf)`.

In [ ]:
import importlib, ex2d2_forward as fw
importlib.reload(fw)

fw.check_verbatim()               # auto-detects the training script; skipped if absent
fw.verify(solver=solver)

## 6 · Cross-evaluation + robustness sweep, statistics

Every design × the four cells of `EVAL_CELLS` (`{linelas, hencky} × {const, tdep}`) × every `T_D`
in `eval_tds` (default `EVAL_TDS = [673, 873, 1073]`; a design's own `T_D` is always solved even if
you shrink the list). `PROP_FOR_CONST` only marks each design family's *native* cell — the
factorial pairing it represents (`linelas+const` / `hencky+tdep`) — in the tables and figures.

Outputs in `results/`:

- `cross_eval.csv` — long format, one row per solve (12 per design), with `TD_design` (training)
  vs `TD` (evaluation) and full solver diagnostics;
- `cross_matrix.csv` — wide, one row per design at its **own** `T_D`: the four `u_out` values plus
  the factorial effects (`dCONST` at each property model, `dPROPS` under each constitutive law, the
  2×2 interaction, native-vs-reference), all as fractions of `u_ref = u(hencky, tdep)`;
- `robust_refcell.csv` — wide, one row per design: `u_out` under the reference cell at every
  evaluation `T_D`, plus the end-to-end span and mid-range curvature;
- `cross_stats.txt` — sections **A–E** (physics cross at the own `T_D`) and **F–G** (robustness
  per design; transferability: at each operating `T_D`, all six design families — constitutive ×
  training `T_D` — ranked under the reference physics);
- figures: `fig_cross_uout.png` (per design, 4 cells at the own `T_D`, hatched = `const`, native
  cell marked ▾), `fig_cross_matrix.png` (seed-averaged 2×2 per `T_D` × family, ±sd),
  `fig_robust_uout.png` (per design, 3 evaluation `T_D` under the reference cell, training `T_D`
  marked ▾), `fig_robust_lines.png` (family-mean `u_out` vs evaluation `T_D`, on-design points
  circled);
- optional `*_fields.npz`: `save_fields='native'` saves fields at the design's **own** `T_D` only;
  `True` additionally saves the off-design temperatures (`*_evalTD<K>_fields.npz`, ~3× the
  files); `False` saves none.

In [ ]:
rows = fw.sweep(root=ROOT, out='results',
                backend=solver.backend,      # reuse the probed backend
                eval_tds=(673, 873, 1073),   # evaluation temperatures (own TD always included)
                n_inc=4,                 # thermal load increments for the hencky NR paths
                save_fields='native',    # fields at the own TD; True -> also at off-design TDs
                verbose=False)           # True prints every NR increment

In [ ]:
_ = fw.report(rows, out='results')
fw.figure(rows, out='results')

from IPython.display import Image, display
for f in ('fig_cross_uout', 'fig_cross_matrix', 'fig_robust_uout', 'fig_robust_lines'):
    display(Image(f'results/{f}.png', width=1100))

## 7 · Increment-independence spot check (optional)

The `hencky` answer must not depend on how the thermal load was ramped. Re-solves one design with 4 / 8 / 16 increments; the spread should be ~1e-13 relative.

In [ ]:
import torch, numpy as np
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
f = sorted(glob.glob(os.path.join(ROOT, 'Run_*hencky*')))[0]
meta = fw.parse_name(os.path.basename(f))
keep, w, m = fw.load_design(f, dev)
mesh = fw.Mesh(keep, dev)
T, T_ip, _ = fw.solve_thermal(mesh, w, fw.CFG, 'tdep', meta['TD'], solver)
vals = {}
for n in (4, 8, 16):
    U, i = fw.solve_mech(mesh, w, T_ip, fw.CFG, 'hencky', 'tdep', meta['TD'],
                         solver, n_inc=n)
    vals[n] = i['u_out']
    print(f"n_inc={n:2d}  u_out = {i['u_out']:.9f}   |R|/|R0| = {i['res_rel']:.1e}")
sp = max(vals.values()) - min(vals.values())
print(f"spread = {sp:.3e}  ({sp/abs(vals[8]):.2e} relative)")

## 8 · Download

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('ex2d2_cross_robust_results', 'zip', 'results')
files.download('ex2d2_cross_robust_results.zip')